In [19]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")

print("Path to dataset files:", path)


Resuming download from 6291456 bytes (62864216 bytes left)...
Resuming download from https://www.kaggle.com/api/v1/datasets/download/mlg-ulb/creditcardfraud?dataset_version_number=3 (6291456/69155672) bytes left.


100%|██████████| 66.0M/66.0M [00:03<00:00, 17.4MB/s]

Extracting files...


Path to dataset files: /Users/flexonafft/.cache/kagglehub/datasets/mlg-ulb/creditcardfraud/versions/3


## Загружаю данные и предобрабатываю

In [ ]:
import pandas as pd
import numpy as np

Datapath = "datasets/creditcard.csv"
df = pd.read_csv(Datapath)

'''
print(df.shape)
display(df.head())
print(df["Class"].value_counts(dropna=False))
print(df.isna().sum().sort_values(ascending=False).head(10))'''

(284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


Class
0    284315
1       492
Name: count, dtype: int64
Time      0
V16       0
Amount    0
V28       0
V27       0
V26       0
V25       0
V24       0
V23       0
V22       0
dtype: int64


In [21]:
df = df.copy()

# Четыре новые фичи (эксперимент)
df["Amount_log1p"] = np.log1p(df["Amount"])
df["Hour"] = ((df["Time"] // 3600) % 24).astype(int)
df["Hour_sin"] = np.sin(2 * np.pi * df["Hour"] / 24)
df["Hour_cos"] = np.cos(2 * np.pi * df["Hour"] / 24)

In [22]:
display(df[["Time", "Hour", "Hour_sin", "Hour_cos", "Amount", "Amount_log1p", "Class"]].head())

,Time,Hour,Hour_sin,Hour_cos,Amount,Amount_log1p,Class
0,0.0,0,0.0,1.0,149.62,5.014760,0
1,0.0,0,0.0,1.0,2.69,1.305626,0
2,1.0,0,0.0,1.0,378.66,5.939276,0
3,1.0,0,0.0,1.0,123.50,4.824306,0
4,2.0,0,0.0,1.0,69.99,4.262539,0


#### Разбиваем данные

In [23]:
from sklearn.model_selection import train_test_split

target = 'Class'
drop_cols = [target]

X = df.drop(columns=drop_cols)
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape, y_train.mean())
print("Test :", X_test.shape, y_test.mean())

Train: (227845, 34) 0.001729245759178389
Test : (56962, 34) 0.0017204452090867595


In [24]:
from catboost import CatBoostClassifier

# Вычислим баланс классов
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
class_weights = [1.0, neg / pos]
print("class_weights:", class_weights)

class_weights: [1.0, np.float64(577.2868020304569)]


In [25]:
model = CatBoostClassifier(iterations=2000, learning_rate=0.05, depth=6,
    loss_function="Logloss", eval_metric="AUC", class_weights=class_weights,
    random_seed=42, verbose=200)

In [26]:
model.fit(X_train, y_train,
eval_set=(X_test, y_test), use_best_model=True)

0:	test: 0.9601927	best: 0.9601927 (0)	total: 69.1ms	remaining: 2m 18s
200:	test: 0.9802038	best: 0.9809705 (131)	total: 2.21s	remaining: 19.7s
400:	test: 0.9815701	best: 0.9821115 (239)	total: 4.17s	remaining: 16.6s
600:	test: 0.9813221	best: 0.9821115 (239)	total: 5.9s	remaining: 13.7s
800:	test: 0.9813971	best: 0.9821115 (239)	total: 7.54s	remaining: 11.3s
1000:	test: 0.9813761	best: 0.9821115 (239)	total: 9.2s	remaining: 9.18s
1200:	test: 0.9813102	best: 0.9821115 (239)	total: 10.8s	remaining: 7.18s
1400:	test: 0.9813102	best: 0.9821115 (239)	total: 12.4s	remaining: 5.31s
1600:	test: 0.9813102	best: 0.9821115 (239)	total: 14s	remaining: 3.49s
1800:	test: 0.9813102	best: 0.9821115 (239)	total: 15.6s	remaining: 1.73s
1999:	test: 0.9813102	best: 0.9821115 (239)	total: 17.2s	remaining: 0us

bestTest = 0.9821114539
bestIteration = 239

Shrink model to first 240 iterations.


## Быстрая проверка качества

In [27]:
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, confusion_matrix

proba = model.predict_proba(X_test)[:, 1]
pred  = (proba >= 0.5).astype(int)

print("ROC-AUC:", roc_auc_score(y_test, proba))
print("PR-AUC :", average_precision_score(y_test, proba))  
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, digits=4))

ROC-AUC: 0.9821114538950075
PR-AUC : 0.8405516075505346
[[56817    47]
 [   13    85]]
              precision    recall  f1-score   support

           0     0.9998    0.9992    0.9995     56864
           1     0.6439    0.8673    0.7391        98

    accuracy                         0.9989     56962
   macro avg     0.8219    0.9333    0.8693     56962
weighted avg     0.9992    0.9989    0.9990     56962

